# FlashAttention руками: online softmax и точное объединение блоков

Вопрос опыта: можно ли получить тот же результат внимания, не сохраняя целиком строку вероятностей? Мы обработаем K/V плитками, сохраняя только максимум m, сумму экспонент l и ненормированный выход o. Это математическое ядро, на котором строится перенос вычислений из HBM ближе к вычислительным блокам.

In [ ]:
import math
import platform

print('python:', platform.python_version())

def dot(left, right):
    return sum(a * b for a, b in zip(left, right))

def assert_close(actual, expected, tolerance=1e-12):
    assert len(actual) == len(expected)
    assert all(abs(a - b) <= tolerance for a, b in zip(actual, expected)), (actual, expected)

queries = [[1.0, 0.0, 0.5, -0.5], [0.0, 1.0, -0.5, 0.25], [0.5, 0.5, 1.0, 0.0]]
keys = [[math.sin(i + j / 3) for j in range(4)] for i in range(1, 10)]
values = [[math.cos(i / 2), math.sin(i / 3), (i - 5) / 5] for i in range(1, 10)]

def attention_scores(query):
    return [dot(query, key) / math.sqrt(len(query)) for key in keys]

## 1. Материальный эталон

Эталон создаёт всю строку scores, затем всю строку вероятностей P и только после этого умножает её на V. Вычитание максимума не меняет softmax и защищает экспоненту от переполнения.

In [ ]:
def naive_attention_from_scores(scores, source_values):
    maximum = max(scores)
    weights = [math.exp(score - maximum) for score in scores]
    denominator = sum(weights)
    probabilities = [weight / denominator for weight in weights]
    return [sum(probability * value[column]
                for probability, value in zip(probabilities, source_values))
            for column in range(len(source_values[0]))]

reference = [naive_attention_from_scores(attention_scores(query), values)
             for query in queries]
for row in reference:
    print([round(value, 6) for value in row])

## 2. Состояние (m, l, o) и обработка плитками

Когда новый блок повышает общий максимум, старые экспоненты были посчитаны в другой шкале. Поэтому и старую сумму l, и старый числитель o нужно умножить на exp(m_old - m_new). После последнего блока деление o/l даёт обычный softmax(scores) @ V.

In [ ]:
def tiled_online_attention(scores, source_values, tile_size):
    m = -math.inf
    l = 0.0
    o = [0.0] * len(source_values[0])

    for start in range(0, len(scores), tile_size):
        tile_scores = scores[start:start + tile_size]
        tile_values = source_values[start:start + tile_size]
        tile_max = max(tile_scores)
        new_m = max(m, tile_max)
        old_scale = 0.0 if m == -math.inf else math.exp(m - new_m)
        tile_weights = [math.exp(score - new_m) for score in tile_scores]
        new_l = old_scale * l + sum(tile_weights)
        new_o = [old_scale * o[column] +
                 sum(weight * value[column]
                     for weight, value in zip(tile_weights, tile_values))
                 for column in range(len(o))]
        m, l, o = new_m, new_l, new_o

    return [value / l for value in o]

for tile_size in (1, 2, 3, 4, 9):
    for query, expected in zip(queries, reference):
        actual = tiled_online_attention(attention_scores(query), values, tile_size)
        assert_close(actual, expected)
print('Все размеры плитки дали тот же результат, что и материальный эталон.')

## 3. Большие логиты и цена промежуточных данных

Online softmax должен оставаться устойчивым при общем большом сдвиге логитов. Для одной строки наивный путь материализует T scores и T probabilities; учебный блочный путь хранит не больше tile_size элементов и состояние фиксированного размера. Настоящее GPU-ядро плиточно обрабатывает также Q и размерность признаков.

In [ ]:
base_scores = attention_scores(queries[0])
huge_scores = [score + 10000.0 for score in base_scores]
expected_huge = naive_attention_from_scores(huge_scores, values)
actual_huge = tiled_online_attention(huge_scores, values, tile_size=3)
assert_close(actual_huge, expected_huge)
assert_close(expected_huge, reference[0])  # softmax не меняется от общего сдвига

T, tile_size, output_width = 8192, 128, 128
naive_intermediate_scalars = 2 * T
tiled_intermediate_scalars = tile_size + 2 + output_width
assert tiled_intermediate_scalars < naive_intermediate_scalars
print('скаляров одной строки, наивно:', naive_intermediate_scalars)
print('верхняя учебная оценка, плиточно:', tiled_intermediate_scalars)

## 4. Намеренная ошибка: забыли изменить шкалу старого блока

Если новый максимум больше прежнего, простое сложение числителей и знаменателей неверно. Ошибка особенно заметна, когда следующие плитки содержат гораздо большие логиты.

In [ ]:
def broken_merge(scores, source_values, tile_size):
    m, l = -math.inf, 0.0
    o = [0.0] * len(source_values[0])
    for start in range(0, len(scores), tile_size):
        tile_scores = scores[start:start + tile_size]
        tile_values = source_values[start:start + tile_size]
        m = max(m, max(tile_scores))
        weights = [math.exp(score - m) for score in tile_scores]
        l += sum(weights)  # ошибка: старое l осталось в прежней шкале
        o = [o[column] + sum(weight * value[column]
             for weight, value in zip(weights, tile_values))
             for column in range(len(o))]
    return [value / l for value in o]

diagnostic_scores = [0.0, 1.0, 10.0, 11.0]
diagnostic_values = [[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0], [0.0, -1.0]]
correct = naive_attention_from_scores(diagnostic_scores, diagnostic_values)
broken = broken_merge(diagnostic_scores, diagnostic_values, tile_size=2)
difference = max(abs(a - b) for a, b in zip(correct, broken))
assert difference > 0.1
print('ошибка объединения обнаружена:', round(difference, 6))

## 5. Переход от формулы к GPU

В ноутбуке keys/values играют роль данных в глобальной памяти, срез плитки — данных, временно перенесённых в более близкую память, а (m, l, o) — состояния строки, удерживаемого рядом с вычислением. FlashAttention добавляет разбиение по нескольким осям, распределение работы между группами потоков, маску, несколько голов, пакет и обратный проход, но не меняет доказанный инвариант объединения.

**Задание.** Добавьте причинную маску и несколько строк Q. Затем сравните число прочитанных/записанных промежуточных скаляров, а не только число арифметических операций.